# Predict Customer Churn ベースライン用Notebook

以下のKaggle Playground Seriesを解くためのベースライン用Notebook
https://www.kaggle.com/competitions/playground-series-s6e3/overview

## コンペ情報
- 評価指標: AUC
- 目的変数: Churn（解約したか否か）
- タスク: 分類問題

## ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, roc_auc_score

import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

## データの読み込み

In [ ]:
p = "/kaggle/input/competitions/playground-series-s6e3/"

df_train = pd.read_csv(p + "train.csv")
df_test = pd.read_csv(p + "test.csv")
df_submission = pd.read_csv(p + "sample_submission.csv")

## データの確認

In [ ]:
print(df_train.shape)
print(df_test.shape)

In [ ]:
df_train.head()

In [ ]:
df_submission.head()

In [ ]:
df_train.info()

In [ ]:
df_train.describe()

In [ ]:
# 欠損値の確認
print("=== train ===")
print(df_train.isnull().sum())
print()
print("=== test ===")
print(df_test.isnull().sum())

## 前処理＆特徴量選択

- ベースラインを作成するうえでのコツ
  - 数値カラム
  - 欠損値が存在しない

In [ ]:
numeric_cols = df_train.select_dtypes(include='number')
numeric_cols

In [ ]:
numeric_cols["tenure"] * numeric_cols["MonthlyCharges"]

In [ ]:
features = ["tenure", "MonthlyCharges"]
target = "Churn"


X_train = df_train[features]
y_train = df_train[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

In [ ]:
X_test = df_test[features]
print("X_test:", X_test.shape)

## モデル作成

In [ ]:
params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,
    "num_leaves": 31,
    "n_estimators": 100000,
    "random_state": 42,
    "importance_type": "gain",
    "verbosity": -1,
}

In [ ]:
n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

metrics = []
imp = pd.DataFrame()
models = []
test_preds = np.zeros(len(X_test)) 


for nfold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n========== Fold {nfold + 1} / {n_splits} ==========")

    x_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    x_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        x_tr, y_tr,
        eval_set=[(x_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    # 学習データ・検証データのスコア
    y_tr_pred = model.predict(x_tr)
    y_va_pred = model.predict(x_va)
    metric_tr = accuracy_score(y_tr, y_tr_pred)
    metric_va = accuracy_score(y_va, y_va_pred)
    print(f"[Accuracy] train: {metric_tr:.4f}, val: {metric_va:.4f}")

    metrics.append([nfold, metric_tr, metric_va])
    models.append(model)
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits


    # 特徴量の重要度を記録
    _imp = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_,
        "fold": nfold,
    })
    imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

## 特徴量の重要度を確認

In [ ]:
metrics = np.array(metrics)

print(f"[CV Result] train: {metrics[:, 1].mean():.4f} ± {metrics[:, 1].std():.4f}")
print(f"[CV Result] val:   {metrics[:, 2].mean():.4f} ± {metrics[:, 2].std():.4f}")

In [ ]:
# 全foldの重要度を平均
imp_mean = imp.groupby("feature")["importance"].mean().sort_values(ascending=False)

# 上位20個をプロット
plt.figure(figsize=(10, 8))
imp_mean.head(20).plot(kind="barh")
plt.xlabel("Importance (gain)")
plt.title("Feature Importance (Top 20)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 提出ファイルの作成

In [ ]:
df_submission

In [ ]:
df_submission[target] = test_preds
df_submission.head()

In [ ]:
df_submission.to_csv("submission.csv", index=False)